# M14 — Conformal Calibration Wrapper on Cross-Task Disagreement Scores

**Model ID:** M14  
**Model Name:** Conformal Prediction Wrapper for Unknown-Detection Threshold  
**Member:** B — Disease Diagnosis & Staged Open-World Learning Lead  
**Project:** OWMTL — Cluster-Aware Open-World Multi-Task Learning for Respiratory Sound and Disease Diagnosis  
**Chunk:** G (Trust & Calibration) — **Selected Novelty Item #2** (`Novelty Search.md` §4.0)  
**Requires:** M2 Backbone (`best_model.pth`), M13 Prototypical Disease Head (`best_model.pth`), Real ICBHI Audio  

---

### Why This Model Exists (Novelty Search §4.0, Attack #6)

**Reviewer Attack #6:** *"AUROC is threshold-free — you need a real operating point and a formal
guarantee, not just a point accuracy."*

**M14's Answer:** Conformal prediction converts M15's continuous disagreement score into a
**distribution-free coverage guarantee**. Given a user-specified miscoverage rate α (e.g., α=0.05
for 95% coverage), the conformal quantile $\hat{q}_{1-\alpha}$ guarantees:

$$P(\text{known patient correctly classified as known}) \geq 1 - \alpha$$

**No parametric assumptions.** Works on any score distribution. Post-hoc — no retraining needed.

### How It Works
1. Compute M15 cross-task disagreement scores on **calibration patients** (known-class held-out).
2. Compute the empirical $(1-\alpha)$-quantile of calibration scores as the conformal threshold.
3. Evaluate on test patients: known patients with scores ≤ threshold are correctly retained;
   unknown patients with scores > threshold are correctly flagged.
4. Sweep α from 0.01 to 0.50 and plot **empirical coverage vs. nominal coverage**.


## Section 1: Setup & Dependencies


In [1]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os
import sys
import re
import json
import math
import time
import glob
import random
import warnings
import datetime
import zipfile
import io
import shutil
import base64
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

warnings.filterwarnings('ignore')

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')
print(f'Python:  {sys.version.split()[0]}')

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 11})
sns.set_style('whitegrid')


Device:  cuda (Tesla T4)
PyTorch: 2.10.0+cu128
Python:  3.12.13


## Section 2: Configuration & Path Resolution (Kaggle & Colab)


In [2]:
# ============================================================
# Section 2: Configuration & Path Resolution (Kaggle & Colab)
# ============================================================

# ---- Auto-detect Platform ----
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M14'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Drive Backup Path: {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- ICBHI Dataset Path Resolution ----
POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    '/content/drive/MyDrive/OWMTL/data/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f'Dynamic Kaggle resolution: {DATA_ROOT}')
            break

# ---- Colab Kaggle auto-download ----
if DATA_ROOT is None and PLATFORM == 'Colab':
    print('\n📥 ICBHI dataset not found. Checking Kaggle credentials...')
    drive_kjson = '/content/drive/MyDrive/kaggle.json'
    if os.path.exists(drive_kjson):
        os.system('mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
    elif not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        try:
            from google.colab import files
            print('Please upload kaggle.json:')
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                os.system('mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
        except Exception: pass

    if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        os.system('pip install -q kaggle')
        os.system('kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --unzip')
        DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f'✅ ICBHI dataset verified: {DATA_ROOT}')
else:
    print(f'⚠️ DATA_ROOT fallback: {DATA_ROOT}')

# ---- Model Checkpoint Resolution ----
def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

M2_CKPT_PATH = resolve_checkpoint([
    '/content/M2_best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
])

if M2_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m2' in f.lower() or 'm2' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M2_CKPT_PATH = os.path.join(root, f)
                print(f'Dynamic Kaggle M2 checkpoint: {M2_CKPT_PATH}')
                break
        if M2_CKPT_PATH: break

M13_CKPT_PATH = resolve_checkpoint([
    '/content/M13_best_model.pth',
    '/kaggle/input/m13-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m13/best_model.pth',
    '/kaggle/input/m13-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M13/best_model.pth',
    '../M13/best_model.pth',
    os.path.join(BASE_DIR, 'results_M13', 'best_model.pth'),
])

if M13_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m13' in f.lower() or 'm13' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M13_CKPT_PATH = os.path.join(root, f)
                print(f'Dynamic Kaggle M13 checkpoint: {M13_CKPT_PATH}')
                break
        if M13_CKPT_PATH: break

CFG = {
    'model_id': 'M14',
    'model_name': 'Conformal Calibration Wrapper — Unknown Detection Threshold',
    'member': 'B',
    'seed': SEED,

    # Shared Audio Parameters (§2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # Classes
    'disease_classes': ['COPD', 'Healthy', 'URTI'],
    'unknown_classes': ['Pneumonia', 'Bronchiectasis', 'Bronchiolitis'],
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],

    # Architecture (must match M2/M13)
    'm2_depth': 5,
    'm2_base_width': 48,
    'proto_embed_dim': 256,
    'proto_temperature': 0.1,

    # Conformal Prediction
    'alpha_values': np.arange(0.01, 0.51, 0.01).tolist(),  # Miscoverage rates to sweep
    'calibration_ratio': 0.5,  # 50% of known-test patients for calibration
    'batch_size': 32,

    'data_root': DATA_ROOT,
    'm2_ckpt_path': M2_CKPT_PATH,
    'm13_ckpt_path': M13_CKPT_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M14'),
}

os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print('M14 CONFIGURATION — Conformal Calibration Wrapper')
print(f"{'='*60}")
print(f"  M2  Checkpoint: {CFG['m2_ckpt_path'] or 'NOT FOUND'}")
print(f"  M13 Checkpoint: {CFG['m13_ckpt_path'] or 'NOT FOUND'}")
print(f"  Data Root:      {CFG['data_root']}")
print(f"  Alpha Sweep:    {len(CFG['alpha_values'])} values (0.01 to 0.50)")
print(f"{'='*60}")


Platform: Kaggle
Dynamic Kaggle resolution: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
✅ ICBHI dataset verified: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
Dynamic Kaggle M2 checkpoint: /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
Dynamic Kaggle M13 checkpoint: /kaggle/input/datasets/barshonbasak/m13-checkpoint/best_model.pth

M14 CONFIGURATION — Conformal Calibration Wrapper
  M2  Checkpoint: /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
  M13 Checkpoint: /kaggle/input/datasets/barshonbasak/m13-checkpoint/best_model.pth
  Data Root:      /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
  Alpha Sweep:    50 values (0.01 to 0.50)


## Section 3: Load ICBHI Audio — Known + Unknown Patients


In [3]:
# ============================================================
# Section 3: Load ICBHI Audio — Known + Unknown Patients
# ============================================================

ICBHI_KNOWN = {'COPD': 0, 'Healthy': 1, 'URTI': 2}
ICBHI_UNKNOWN = {'Pneumonia', 'Bronchiectasis', 'Bronchiolitis'}

try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start, duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        # A silent all-zero spectrogram here would be trained on and
        # scored as a real cycle. Fail instead of substituting
        # (Model_Training_Protocol.md section 1.2).
        raise RuntimeError(f"failed to load audio: {wav_path}") from e
    if len(audio) == 0:
        # Empty decode is a failed read, not a silent zero cycle.
        raise RuntimeError(f"empty audio decoded from audio: {wav_path}")
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError: continue
            if end <= start: continue
            if crackle == 0 and wheeze == 0: label = 0
            elif crackle == 1 and wheeze == 0: label = 1
            elif crackle == 0 and wheeze == 1: label = 2
            else: label = 3
            cycles.append({'start': start, 'end': end, 'label': label})
    return cycles

def load_diagnosis_map(data_root):
    target_names = ['patient_diagnosis.csv', 'ICBHI_Challenge_diagnosis.txt', 'patient_diagnosis.txt']
    candidates = []
    curr = data_root
    for _ in range(4):
        for name in target_names: candidates.append(os.path.join(curr, name))
        parent = os.path.dirname(curr)
        if parent == curr: break
        curr = parent
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            for name in target_names:
                if name in files: candidates.append(os.path.join(root, name))
    for path in candidates:
        if not os.path.exists(path): continue
        diag_map = {}
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line_str = line.strip()
                if not line_str: continue
                parts = [p.strip() for p in re.split(r'[,;\t\s]+', line_str) if p.strip()]
                if len(parts) >= 2:
                    try:
                        pid = int(parts[0])
                        diag_map[pid] = parts[1]
                    except ValueError: continue
        if diag_map:
            print(f'Loaded diagnosis map: {path} ({len(diag_map)} patients)')
            return diag_map
    return None

def build_conformal_datasets(data_root, cfg):
    """Load all known + unknown patient cycles for conformal calibration."""
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')
    diag_map = load_diagnosis_map(data_root)
    if diag_map is None:
        raise FileNotFoundError('Diagnosis map not found')

    known_rows, unknown_rows = [], []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        disease = diag_map.get(pid)
        if disease is None: continue
        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            row = {
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'sound_label': c['label'],
                'disease_name': disease
            }
            if disease in ICBHI_KNOWN:
                row['disease_label'] = ICBHI_KNOWN[disease]
                row['is_known'] = True
                known_rows.append(row)
            elif disease in ICBHI_UNKNOWN:
                row['disease_label'] = -1  # Unknown
                row['is_known'] = False
                unknown_rows.append(row)

    df_known = pd.DataFrame(known_rows)
    df_unknown = pd.DataFrame(unknown_rows)

    # Patient-independent split for Known: 60% train, 20% calibration, 20% test
    known_pids = sorted(df_known['patient_id'].unique())
    np.random.seed(SEED)
    np.random.shuffle(known_pids)

    n_train = int(len(known_pids) * 0.6)
    n_cal = int(len(known_pids) * 0.2)
    train_pids = set(known_pids[:n_train])
    cal_pids = set(known_pids[n_train:n_train + n_cal])
    test_pids = set(known_pids[n_train + n_cal:])

    df_known_train = df_known[df_known['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_known_cal = df_known[df_known['patient_id'].isin(cal_pids)].reset_index(drop=True)
    df_known_test = df_known[df_known['patient_id'].isin(test_pids)].reset_index(drop=True)

    return df_known_train, df_known_cal, df_known_test, df_unknown

class RealICBHI_Dataset(Dataset):
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        label = row.get('disease_label', -1)
        return (torch.from_numpy(spec),
                torch.tensor(label, dtype=torch.long),
                row['patient_id'])

print('\n--- LOADING REAL ICBHI AUDIO FOR CONFORMAL CALIBRATION ---')
df_known_train, df_known_cal, df_known_test, df_unknown = build_conformal_datasets(CFG['data_root'], CFG)

print(f'Known Train (Prototype Computation):  {len(df_known_train)} cycles ({df_known_train["patient_id"].nunique()} patients)')
print(f'Known Calibration (Conformal):        {len(df_known_cal)} cycles ({df_known_cal["patient_id"].nunique()} patients)')
print(f'Known Test (Coverage Eval):           {len(df_known_test)} cycles ({df_known_test["patient_id"].nunique()} patients)')
print(f'Unknown (OOD Evaluation):             {len(df_unknown)} cycles ({df_unknown["patient_id"].nunique()} patients)')

known_train_ds = RealICBHI_Dataset(df_known_train, CFG)
known_cal_ds = RealICBHI_Dataset(df_known_cal, CFG)
known_test_ds = RealICBHI_Dataset(df_known_test, CFG)
unknown_ds = RealICBHI_Dataset(df_unknown, CFG)

train_loader = DataLoader(known_train_ds, batch_size=CFG['batch_size'], shuffle=False)
cal_loader = DataLoader(known_cal_ds, batch_size=CFG['batch_size'], shuffle=False)
test_loader = DataLoader(known_test_ds, batch_size=CFG['batch_size'], shuffle=False)
unknown_loader = DataLoader(unknown_ds, batch_size=CFG['batch_size'], shuffle=False)



--- LOADING REAL ICBHI AUDIO FOR CONFORMAL CALIBRATION ---
Loaded diagnosis map: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv (126 patients)
Known Train (Prototype Computation):  2927 cycles (62 patients)
Known Calibration (Conformal):        1444 cycles (20 patients)
Known Test (Coverage Eval):           1940 cycles (22 patients)
Unknown (OOD Evaluation):             549 cycles (19 patients)


## Section 4: Architecture & Checkpoint Loading (M2 + M13)


In [4]:
# ============================================================
# Section 4: Architecture & Checkpoint Loading (M2 + M13)
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.head(self.dropout(feat))
    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)

class PrototypicalDiseaseHead(nn.Module):
    def __init__(self, input_dim, embed_dim=256, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self.projection = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )
        self.embed_dim = embed_dim
    def project(self, embeddings):
        z = self.projection(embeddings)
        return F.normalize(z, p=2, dim=-1)
    def compute_prototypes(self, support_embeddings, support_labels):
        prototypes = torch.zeros(self.num_classes, self.embed_dim, device=support_embeddings.device)
        for c in range(self.num_classes):
            mask = (support_labels == c)
            if mask.sum() > 0:
                prototypes[c] = support_embeddings[mask].mean(dim=0)
        return F.normalize(prototypes, p=2, dim=-1)

def smart_load_checkpoint(path, device):
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                target = 'best_model.pth'
                if target not in names:
                    target = next((n for n in names if n.endswith('.pth')), None)
                if target:
                    print(f'📦 Extracting {target} from zip bundle {path}')
                    with z.open(target) as f:
                        return torch.load(io.BytesIO(f.read()), map_location=device, weights_only=False)
        except Exception as e:
            print(f'Zip extraction note: {e}')
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception:
        return torch.load(path, map_location=device, weights_only=True)

# Initialize & load
backbone = M2_CNN(num_classes=4, depth=CFG['m2_depth'], base_width=CFG['m2_base_width']).to(DEVICE)
proto_head = PrototypicalDiseaseHead(input_dim=backbone.embedding_dim, embed_dim=CFG['proto_embed_dim'], num_classes=3).to(DEVICE)

m2_loaded = False
if CFG['m2_ckpt_path']:
    try:
        ckpt = smart_load_checkpoint(CFG['m2_ckpt_path'], DEVICE)
        sd = ckpt.get('model_state', ckpt)
        if isinstance(sd, dict) and 'model_state_dict' in sd: sd = sd['model_state_dict']
        backbone.load_state_dict(sd, strict=False)
        print(f'✅ Loaded M2 Backbone from {CFG["m2_ckpt_path"]}')
        m2_loaded = True
    except Exception as e:
        print(f'⚠️ M2 load failed: {e}')
if not m2_loaded:
    print('⚠️ Using default M2 weights')

m13_loaded = False
if CFG['m13_ckpt_path']:
    try:
        ckpt13 = smart_load_checkpoint(CFG['m13_ckpt_path'], DEVICE)
        if isinstance(ckpt13, dict) and 'model_state' in ckpt13:
            proto_head.load_state_dict(ckpt13['model_state'], strict=False)
        elif isinstance(ckpt13, dict):
            proto_head.load_state_dict(ckpt13, strict=False)
        print(f'✅ Loaded M13 Prototypical Head from {CFG["m13_ckpt_path"]}')
        m13_loaded = True
    except Exception as e:
        print(f'⚠️ M13 load failed: {e}')
if not m13_loaded:
    print('⚠️ Using default M13 weights')

backbone.eval()
proto_head.eval()
for p in backbone.parameters(): p.requires_grad = False
for p in proto_head.parameters(): p.requires_grad = False

print(f'\nM2 Backbone embedding dim: {backbone.embedding_dim}')
print(f'Proto Head params: {sum(p.numel() for p in proto_head.parameters()):,}')


✅ Loaded M2 Backbone from /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
✅ Loaded M13 Prototypical Head from /kaggle/input/datasets/barshonbasak/m13-checkpoint/best_model.pth

M2 Backbone embedding dim: 768
Proto Head params: 526,080


## Section 5: Compute Cross-Task Disagreement Scores

The disagreement score combines:
1. **Prototype distance:** How far the sample embedding is from the nearest known-class prototype (M13).
2. **Sound-event entropy:** How uncertain the sound-event head (M2) is about the respiratory event type.

Higher disagreement → more likely to be an unknown disease.


In [5]:
# ============================================================
# Section 5: Compute Cross-Task Disagreement Scores
# ============================================================
#
# The disagreement score measures how inconsistent the sound-event
# head's implied disease profile is versus the prototypical disease
# head's actual prediction. Higher scores = more anomalous.

def compute_prototypes(backbone, proto_head, loader, device, num_classes=3):
    """Compute class prototypes from training data."""
    all_z, all_y = [], []
    with torch.no_grad():
        for specs, labels, pids in loader:
            embeds = backbone.get_embedding(specs.to(device))
            z = proto_head.project(embeds)
            all_z.append(z)
            all_y.append(labels.to(device))
    all_z = torch.cat(all_z, 0)
    all_y = torch.cat(all_y, 0)
    prototypes = torch.zeros(num_classes, proto_head.embed_dim, device=device)
    for c in range(num_classes):
        mask = (all_y == c)
        if mask.sum() > 0:
            prototypes[c] = all_z[mask].mean(dim=0)
    return F.normalize(prototypes, p=2, dim=-1)

def compute_disagreement_scores(backbone, proto_head, prototypes, loader, device):
    """
    Cross-task disagreement score:
    1. Sound-event head softmax → implied disease profile (via learned mapping)
    2. Prototypical head → nearest-prototype distance
    3. Disagreement = min_prototype_distance (high = far from all known prototypes = unknown)
    """
    scores, pids_out = [], []
    with torch.no_grad():
        for specs, labels, pids in loader:
            embeds = backbone.get_embedding(specs.to(device))
            z = proto_head.project(embeds)
            # Distance to nearest prototype (L2 squared)
            dists = torch.cdist(z, prototypes, p=2) ** 2  # [B, num_classes]
            min_dist = dists.min(dim=-1).values  # [B] — distance to closest prototype

            # Sound-event head logits for cross-task signal
            se_logits = backbone(specs.to(device))  # [B, 4]
            se_entropy = -(F.softmax(se_logits, dim=-1) * F.log_softmax(se_logits, dim=-1)).sum(dim=-1)  # [B]

            # Combined disagreement: prototype distance + sound-event uncertainty
            disagreement = min_dist + se_entropy

            scores.extend(disagreement.cpu().numpy().tolist())
            pids_out.extend([int(p) for p in pids])

    return np.array(scores), np.array(pids_out)

# Compute prototypes from training set
print('Computing class prototypes from training data...')
prototypes = compute_prototypes(backbone, proto_head, train_loader, DEVICE)

# Compute disagreement scores for each split
print('Computing disagreement scores...')
cal_scores, cal_pids = compute_disagreement_scores(backbone, proto_head, prototypes, cal_loader, DEVICE)
test_scores, test_pids = compute_disagreement_scores(backbone, proto_head, prototypes, test_loader, DEVICE)
unk_scores, unk_pids = compute_disagreement_scores(backbone, proto_head, prototypes, unknown_loader, DEVICE)

# Aggregate to patient level (mean score per patient)
def aggregate_patient_scores(scores, pids):
    patient_scores = {}
    for s, p in zip(scores, pids):
        if p not in patient_scores:
            patient_scores[p] = []
        patient_scores[p].append(s)
    return {p: np.mean(v) for p, v in patient_scores.items()}

cal_patient_scores = aggregate_patient_scores(cal_scores, cal_pids)
test_patient_scores = aggregate_patient_scores(test_scores, test_pids)
unk_patient_scores = aggregate_patient_scores(unk_scores, unk_pids)

cal_vals = np.array(list(cal_patient_scores.values()))
test_vals = np.array(list(test_patient_scores.values()))
unk_vals = np.array(list(unk_patient_scores.values()))

print(f'\n--- DISAGREEMENT SCORE STATISTICS ---')
print(f'  Calibration (known):  mean={np.mean(cal_vals):.4f}, std={np.std(cal_vals):.4f}, n={len(cal_vals)} patients')
print(f'  Test (known):         mean={np.mean(test_vals):.4f}, std={np.std(test_vals):.4f}, n={len(test_vals)} patients')
print(f'  Unknown (OOD):        mean={np.mean(unk_vals):.4f}, std={np.std(unk_vals):.4f}, n={len(unk_vals)} patients')


Computing class prototypes from training data...
Computing disagreement scores...

--- DISAGREEMENT SCORE STATISTICS ---
  Calibration (known):  mean=0.8537, std=0.1415, n=20 patients
  Test (known):         mean=0.9200, std=0.1414, n=22 patients
  Unknown (OOD):        mean=0.9043, std=0.1183, n=19 patients


## Section 6: Conformal Calibration & Coverage Sweep

**Split-conformal prediction** (Vovk et al., 2005):
- Uses calibration set scores to compute a threshold $\hat{q}_{1-\alpha}$.
- **Guarantee:** $P(\text{known patient correctly retained}) \geq 1 - \alpha$.
- No parametric assumptions on the score distribution.
- Sweeps α from 0.01 to 0.50 to show coverage-detection trade-off.


In [6]:
# ============================================================
# Section 6: Conformal Calibration & Coverage Sweep
# ============================================================
#
# Split-conformal prediction (Vovk et al., 2005):
# Given calibration scores {s_1, ..., s_n} from known patients,
# the conformal quantile at level (1 - alpha) is:
#   q_hat = ceil((n+1)(1-alpha)) / n -th order statistic
# A test sample with score <= q_hat is classified as 'known'.

def conformal_quantile(cal_scores, alpha):
    """Compute the (1-alpha) conformal quantile."""
    n = len(cal_scores)
    sorted_scores = np.sort(cal_scores)
    idx = int(np.ceil((n + 1) * (1 - alpha))) - 1
    idx = min(max(idx, 0), n - 1)
    return sorted_scores[idx]

# Sweep alpha values
alpha_values = CFG['alpha_values']
coverage_results = []

for alpha in alpha_values:
    q_hat = conformal_quantile(cal_vals, alpha)

    # Known-test coverage: fraction of known patients correctly retained
    known_retained = (test_vals <= q_hat).sum()
    empirical_coverage = known_retained / len(test_vals)

    # Unknown detection: fraction of unknown patients flagged
    unk_flagged = (unk_vals > q_hat).sum()
    unknown_detection_rate = unk_flagged / len(unk_vals)

    coverage_results.append({
        'alpha': round(float(alpha), 4),
        'nominal_coverage': round(float(1 - alpha), 4),
        'conformal_threshold': round(float(q_hat), 6),
        'empirical_coverage': round(float(empirical_coverage), 4),
        'unknown_detection_rate': round(float(unknown_detection_rate), 4),
        'known_retained': int(known_retained),
        'known_total': int(len(test_vals)),
        'unknown_flagged': int(unk_flagged),
        'unknown_total': int(len(unk_vals)),
    })

# Key operating points
for target_cov in [0.95, 0.90, 0.80]:
    target_alpha = 1 - target_cov
    best = min(coverage_results, key=lambda r: abs(r['alpha'] - target_alpha))
    print(f'\nα={best["alpha"]:.2f} → Nominal Coverage={best["nominal_coverage"]:.0%}:')
    print(f'  Conformal Threshold:    {best["conformal_threshold"]:.4f}')
    print(f'  Empirical Coverage:     {best["empirical_coverage"]:.1%} ({best["known_retained"]}/{best["known_total"]} known patients retained)')
    print(f'  Unknown Detection Rate: {best["unknown_detection_rate"]:.1%} ({best["unknown_flagged"]}/{best["unknown_total"]} unknowns flagged)')

# Full AUROC on combined test + unknown
all_scores = np.concatenate([test_vals, unk_vals])
all_labels = np.concatenate([np.zeros(len(test_vals)), np.ones(len(unk_vals))])  # 0=known, 1=unknown
auroc = roc_auc_score(all_labels, all_scores)
aupr = average_precision_score(all_labels, all_scores)

print(f'\n--- OVERALL DETECTION METRICS ---')
print(f'  AUROC (patient-level): {auroc:.4f}')
print(f'  AUPR  (patient-level): {aupr:.4f}')



α=0.05 → Nominal Coverage=95%:
  Conformal Threshold:    1.1101
  Empirical Coverage:     86.4% (19/22 known patients retained)
  Unknown Detection Rate: 10.5% (2/19 unknowns flagged)

α=0.10 → Nominal Coverage=90%:
  Conformal Threshold:    0.9947
  Empirical Coverage:     72.7% (16/22 known patients retained)
  Unknown Detection Rate: 21.1% (4/19 unknowns flagged)

α=0.20 → Nominal Coverage=80%:
  Conformal Threshold:    0.9921
  Empirical Coverage:     72.7% (16/22 known patients retained)
  Unknown Detection Rate: 21.1% (4/19 unknowns flagged)

--- OVERALL DETECTION METRICS ---
  AUROC (patient-level): 0.4522
  AUPR  (patient-level): 0.4351


## Section 7: Visualization — Coverage Plots


In [7]:
# ============================================================
# Section 7: Visualization — Coverage Plots
# ============================================================

# Plot 1: Empirical vs Nominal Coverage
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

nominal = [r['nominal_coverage'] for r in coverage_results]
empirical = [r['empirical_coverage'] for r in coverage_results]
detection = [r['unknown_detection_rate'] for r in coverage_results]

ax = axes[0]
ax.plot(nominal, empirical, 'b-o', markersize=3, lw=2, label='Empirical Coverage')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Ideal (empirical = nominal)')
ax.fill_between(nominal, empirical, [n for n in nominal], alpha=0.1, color='blue')
ax.set_xlabel('Nominal Coverage (1 - α)')
ax.set_ylabel('Empirical Coverage')
ax.set_title('M14 — Conformal Coverage Validity\n(Above diagonal = conservative)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.5, 1.05)

# Plot 2: Unknown Detection Rate vs Nominal Coverage
ax = axes[1]
ax.plot(nominal, detection, 'r-s', markersize=3, lw=2, label='Unknown Detection Rate')
ax.set_xlabel('Nominal Coverage (1 - α)')
ax.set_ylabel('Unknown Detection Rate')
ax.set_title('M14 — Unknown Detection vs Coverage\n(Trade-off: higher coverage ↔ lower detection)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0, 1.05)

# Plot 3: Score distributions
ax = axes[2]
ax.hist(cal_vals, bins=30, alpha=0.5, label='Known (Calibration)', color='blue', density=True)
ax.hist(unk_vals, bins=30, alpha=0.5, label='Unknown (OOD)', color='red', density=True)
# Mark 95% conformal threshold
q95 = conformal_quantile(cal_vals, 0.05)
ax.axvline(q95, color='green', linestyle='--', lw=2, label=f'95% Conformal Threshold ({q95:.3f})')
ax.set_xlabel('Disagreement Score')
ax.set_ylabel('Density')
ax.set_title('M14 — Score Distribution\n(Known vs Unknown Patients)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
for d in sorted({CFG['results_dir'], BASE_DIR}):
    fig.savefig(os.path.join(d, 'conformal_coverage.png'), dpi=150, bbox_inches='tight')
print('Saved: conformal_coverage.png')
plt.show()
plt.close()


Saved: conformal_coverage.png


## Section 8: Generate results_M14.json


In [8]:
# ============================================================
# Section 8: Generate results_M14.json (§4 Schema)
# ============================================================

# Find the 95% operating point
op95 = min(coverage_results, key=lambda r: abs(r['alpha'] - 0.05))
op90 = min(coverage_results, key=lambda r: abs(r['alpha'] - 0.10))

results = {
    'meta': {
        'model_id': 'M14',
        'model_name': 'Conformal Calibration Wrapper — Unknown Detection',
        'member': 'B',
        'member_name': 'Member B (Disease Diagnosis & OWL)',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'Conformal prediction wrapper on M15 cross-task disagreement scores. '
            'Distribution-free coverage guarantee (Vovk et al., 2005). '
            'No retraining — post-hoc calibration on known-class calibration patients. '
            'Selected Novelty Item #2 (Novelty Search §4.0, answers Reviewer Attack #6). '
            'Evaluated at patient-level on real ICBHI audio.'
        ),
    },
    'config': {k: v for k, v in CFG.items() if not callable(v) and not isinstance(v, np.ndarray)},
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'known_train_patients': int(df_known_train['patient_id'].nunique()),
        'known_cal_patients': int(df_known_cal['patient_id'].nunique()),
        'known_test_patients': int(df_known_test['patient_id'].nunique()),
        'unknown_patients': int(df_unknown['patient_id'].nunique()),
        'known_classes': CFG['disease_classes'],
        'unknown_classes': CFG['unknown_classes'],
        'split_method': 'patient_independent_60_20_20',
    },
    'best_metrics': {
        'auroc': round(float(auroc), 4),
        'aupr': round(float(aupr), 4),
        'conformal_threshold_95': round(float(op95['conformal_threshold']), 6),
        'empirical_coverage_95': round(float(op95['empirical_coverage']), 4),
        'unknown_detection_rate_95': round(float(op95['unknown_detection_rate']), 4),
        'conformal_threshold_90': round(float(op90['conformal_threshold']), 6),
        'empirical_coverage_90': round(float(op90['empirical_coverage']), 4),
        'unknown_detection_rate_90': round(float(op90['unknown_detection_rate']), 4),
    },
    'conformal_sweep': coverage_results,
    'baseline_comparisons': {
        'm15_auroc': 0.5782,
        'm29_energy_auroc': 0.6466,
        'reference_m6_openmax_auroc': 0.4516,
    },
    'ablation': {
        'ablation_group': 'calibration_method',
        'ablation_role': 'primary_novelty',
        'baseline_model_id': 'M15',
        'variable_changed': 'conformal_prediction_wrapper_on_disagreement_scores',
        'variables_held_constant': [
            'backbone: M2_CNN (FROZEN)',
            'disease_head: M13_prototypical (FROZEN)',
            'disagreement_score: M15 cross-task mechanism',
            'data_split: patient_independent_60_20_20',
            'seed: 42',
        ],
        'component_flags': {
            'has_sound_event_head': True,
            'has_disease_head': True,
            'has_cross_task_consistency': True,
            'has_conformal_calibration': True,
            'has_cqkd_regularization': False,
            'has_openmax_rejection': False,
            'owl_stage': 1,
            'compression_clusters': None,
        },
        'loss_weights': {
            'sound_event_weight': None,
            'disease_weight': None,
            'consistency_weight': None,
        },
    },
}

for out_dir in sorted({CFG['results_dir'], BASE_DIR}):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M14.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'✅ Saved: {rpath}')

print(f'\n{"="*60}')
print('M14 CONFORMAL CALIBRATION SUMMARY')
print(f'{"="*60}')
print(f'  Patient-Level AUROC:        {auroc:.4f}')
print(f'  Patient-Level AUPR:         {aupr:.4f}')
print(f'  95% Conformal Threshold:    {op95["conformal_threshold"]:.4f}')
print(f'    Empirical Coverage:       {op95["empirical_coverage"]:.1%}')
print(f'    Unknown Detection Rate:   {op95["unknown_detection_rate"]:.1%}')
print(f'  90% Conformal Threshold:    {op90["conformal_threshold"]:.4f}')
print(f'    Empirical Coverage:       {op90["empirical_coverage"]:.1%}')
print(f'    Unknown Detection Rate:   {op90["unknown_detection_rate"]:.1%}')
print(f'{"="*60}')


✅ Saved: /kaggle/working/results_M14.json
✅ Saved: /kaggle/working/results_M14/results_M14.json

M14 CONFORMAL CALIBRATION SUMMARY
  Patient-Level AUROC:        0.4522
  Patient-Level AUPR:         0.4351
  95% Conformal Threshold:    1.1101
    Empirical Coverage:       86.4%
    Unknown Detection Rate:   10.5%
  90% Conformal Threshold:    0.9947
    Empirical Coverage:       72.7%
    Unknown Detection Rate:   21.1%


## Section 9: Bundle & Download


In [9]:
# ============================================================
# Section 9: Bundle & Download Output Files (Kaggle & Colab)
# ============================================================
from IPython.display import display, HTML, FileLink

zip_name = 'M14_results_bundle'
zip_path = os.path.join(BASE_DIR, zip_name)
if os.path.exists(zip_path + '.zip'): os.remove(zip_path + '.zip')

archive = shutil.make_archive(zip_path, 'zip', CFG['results_dir'])
size_mb = os.path.getsize(archive) / (1024 * 1024)

print(f"\n{'='*60}")
print('M14 RESULTS DOWNLOAD BUNDLE')
print(f"{'='*60}")
print(f'Zip: {archive} ({size_mb:.2f} MB)')

if PLATFORM == 'Kaggle':
    print('\n📥 Kaggle Clickable Download Link:')
    display(FileLink('M14_results_bundle.zip'))

try:
    with open(archive, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    href = f'data:application/zip;base64,{b64}'
    html = f'''
<div style="background:#e7f5ff;border:1px solid #74c0fc;padding:16px;border-radius:8px;margin:12px 0;">
  <h3 style="margin-top:0;color:#1864ab;">📥 M14 Results Bundle ({size_mb:.2f} MB)</h3>
  <a href="{href}" download="M14_results_bundle.zip"
     style="display:inline-block;background:#1c7ed6;color:white;padding:12px 24px;
            text-decoration:none;border-radius:6px;font-weight:bold;">⬇️ Download M14_results_bundle.zip</a>
</div>'''
    display(HTML(html))
except Exception as e:
    print(f'Download note: {e}')



M14 RESULTS DOWNLOAD BUNDLE
Zip: /kaggle/working/M14_results_bundle.zip (0.13 MB)

📥 Kaggle Clickable Download Link:


/kaggle/working/M14_results_bundle.zip